---
## 🎁 가산점

### A. 데이터의 다양성
- NTP ICE 내 다양한 데이터셋 모두 활용 가능. (https://ice.ntp.niehs.nih.gov/DATASETDESCRIPTION)
### B. Feature(descriptor)의 다양성
- rdkit, VEGA, 등
### 💬 추가 설명 (자유 기술)

# 기말고사 Template 1 — Data Pipeline

**이름:** ______________ &nbsp; **학번:** ______________ &nbsp;

---

## 📋 채점 기준 (총 50점)

| 항목 | 배점 | 채점 포인트 |
|---|---|---|
| **1. 데이터 분포 파악 및 전처리** | 15점 | 모델 개발 전, 중복 화합물 체크, smiles 코드 정리 등 모델 개발 전 확인해야 할 사항들을 확인. |
| **2. Descriptor 계산** | 15점 | 모델 개발에 사용할 descriptor의 다양성 |
| **3. 데이터 시각화 자료** | 15점 | 구조 분포, 라벨 비율 등 데이터 현황을 시각화한 자료 |
| **4. 코드 가독성 & 주석** | 5점 | 변수의 의미와 코드의 간결성을 평가. |

#### A. 데이터 소스의 다양성
- NTP ICE에서 구할 수 있는 다양한 데이터
- NTP ICE 외 추가 데이터 확보

## 📁 입력 / 출력 예시
- **입력**: `skin_irritation.xlsx` (NTP ICE) + (선택) 외부 데이터
- **출력**: `final_dataset_descriptors.csv`  (Chemical_Name, SMILES, label, 2D descriptor [+ fingerprint 등])

In [ ]:
import pandas as pd
import numpy as np

# ==========================================
# 1. 데이터 불러오기 및 전처리 (배점: 15점)
# ==========================================
print("1. 데이터 불러오기 및 전처리 시작...")
file_path = "acute_oral.xlsx"
df_raw = pd.read_excel(file_path, sheet_name="Data")
print(f"[Data] 원본 데이터 크기: {df_raw.shape}")

# SMILES 결측치 행 제거
df_clean = df_raw[df_raw["SMILES"].notna()].copy()
print(f"[Data] SMILES 결측치 제거 후 크기: {df_clean.shape}")

# 급성 경구 독성(Acute Oral Toxicity) 분석을 위해 LD50 Endpoint 필터링
df_ld50 = df_clean[df_clean["Endpoint"] == "LD50"].copy()
print(f"[Data] LD50 Endpoint 데이터 크기: {df_ld50.shape}")

# Response(LD50 수치)를 수치형 데이터로 변환
df_ld50["Response"] = pd.to_numeric(df_ld50["Response"], errors='coerce')
df_ld50 = df_ld50[df_ld50["Response"].notna()]
print(f"[Data] 수치형 변환 후 데이터 크기: {df_ld50.shape}")

# 중복 화합물 처리: 동일 SMILES에 대해 LD50 중앙값(median)을 사용하여 대표값으로 선정
df_unique = df_ld50.groupby("SMILES").agg({
    "Chemical_Name": "first",
    "Response": "median"
}).reset_index()
print(f"[Data] 중복 화합물 제거 후 유니크 화합물 수: {df_unique.shape[0]}")

# GHS 분류 기준에 따른 이진 분류 라벨 정의
# LD50 <= 2000 mg/kg은 독성 물질(label=1), LD50 > 2000 mg/kg은 비독성 물질(label=0)로 분류
threshold = 2000
df_unique["label"] = (df_unique["Response"] <= threshold).astype(int)
print("\n[label] 클래스 라벨 분포:")
print(df_unique["label"].value_counts())

print("\n[Data Sample] 처음 5개 행 데이터:")
print(df_unique[["Chemical_Name", "SMILES", "Response", "label"]].head())
